In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/My Drive')

Mounted at /content/drive


In [ ]:
!pip install PyWavelets

In [ ]:
import numpy as np
import pandas as pd
import random

from matplotlib import pyplot as plt
import time

import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
import torch.nn as nn
from tqdm import tqdm
from torch.optim import lr_scheduler
from math import exp

import os

In [ ]:
os.makedirs('./data/Teamdata/standard_wells/minish_area',exist_ok=True)
os.makedirs('./data/Teamdata/standard_wells/minish_area',exist_ok=True)

dfTrain=pd.read_csv('/content/drive/MyDrive/logcompletion/data/Teamdata/standard_wells/minish_area/'+f'mini_train_standard.csv',engine='python',sep=';')
dfTest=pd.read_csv('/content/drive/MyDrive/logcompletion/data/Teamdata/standard_wells/minish_area/'+f'mini_test_standard.csv',engine='python',sep=';')



"""
loc_li是每口井的完整位置，maskc是两口井连接处sam_len长度为0其余为1的掩码
获得train_maskc和train_loc_li是为了避免mask0跨越两口井的特殊情况，（所有井都是拉在一个维度的，可能会出现跨越井的情况）
"""
sam_len=512
# 得到train_maskc和train_loc_li
train_well = dfTrain['WELL'].copy()  #注意只取一列则现在well是Series而不是DataFrame
train_maskc = np.ones((dfTrain.shape[0],1))   #maskc定义为mask_cat_loc

train_wells_name = dfTrain['WELL'].unique()
train_wells_num = len(train_wells_name)
train_loc_li = [-1]  #loc_li存储完井位置,加上-1为表示起始处，0不可，因为起始点可能选择到0
for i in range(train_wells_num):
    loc = dfTrain[dfTrain['WELL']==train_wells_name[i]].index[-1].tolist()  #index取[-1]即为该井完井位置
    #print('loc',loc)
    train_loc_li.append(loc)
    train_maskc[loc-sam_len+1 : loc+1]=0
# 得到test_maskc和test_loc_li
test_well = dfTest['WELL'].copy()  #注意只取一列则现在well是Series而不是DataFrame
test_maskc = np.ones((dfTest.shape[0],1))   #maskc定义为mask_cat_loc

test_wells_name = dfTest['WELL'].unique()
#print(test_wells_name[0].type)  #int64
test_wells_num = len(test_wells_name)
test_loc_li = [-1]  #loc_li存储完井位置
for i in range(test_wells_num):
    loc = dfTest[dfTest['WELL']==test_wells_name[i]].index[-1].tolist()  #index取[-1]即为该井完井位置
    #print('loc',loc)
    test_loc_li.append(loc)
    test_maskc[loc-sam_len+1 : loc+1]=0



In [ ]:
from typing import Union, Tuple, Optional

from torch.autograd import Variable
from torch.utils.data import Dataset
from torch import Tensor
from torch.nn import Parameter

import math

class Data:
    def __init__(self, input_sam, label):
        self.x = input_sam
        self.y = label
    def to(self, device):
        self.x = self.x.to(device)
        self.y = self.y.to(device)
        return self

class my_dataset(Dataset):
    def __init__(self, data, need_maskd, maskc, loc_li, fea_litho, sam_num, sam_len):
        self.signal = data  #这里data和self.signal是dataframe
        self.sam_num = sam_num
        self.sam_len = sam_len
        self.fea_litho = fea_litho
        self.need_maskd = need_maskd
        self.maskc = maskc
        self.loc_li = loc_li

    def __getitem__(self, idx):
        label, mask0 = self.label_mask0(self.signal)

        if self.need_maskd == True:
            maskd=self.mask_d(label,mask0)
        else:
            maskd = np.ones((label.shape[0],label.shape[1]))

        #制作对应的sample
        input=label*maskd
        x = input.T

        x = torch.tensor(x, dtype=torch.float)
        label = torch.tensor(label.T, dtype=torch.float)
        return Data(input_sam=x,label=label)

    def __len__(self):
        return self.sam_num

    def depth_loc(self,a):
        b = int((a-136.086)/0.152)  #元坝起始点深度3618.0，0.125；原norway136.086，0.152
        return b

    def label_mask0(self,df):
        '''
        选取随机的起始点，
        制作单个label和对应的初始mask0，该mask0参与loss，不需要恢复
        '''
        awmd = df['DEPTH_MD'].copy()   #awmd is all_wells_md
        awfea = df[self.fea_litho].copy()   #awfea is all_wells_fea
        #print('awfea.shape[0]',awfea.shape[0])
        #awfea = self.precond(awfea)

        #选取随机起始点
        stap=np.random.randint(0,len(awfea)-self.sam_len)  #stap表示start_point
        #不选拼接处的数据
        while(self.maskc[stap]==0):
            stap=np.random.randint(0,len(awfea)-self.sam_len)
        endp=stap+self.sam_len  #endp表示end_point

        i=0
        while(self.loc_li[i]<stap):   # and stap<loc_li[i+1]
            #print('self.loc_li[i]',self.loc_li[i])
            if stap<self.loc_li[i+1]:
                sam_stap = self.depth_loc(awmd.iloc[stap])
                sam_endp = self.depth_loc(awmd.iloc[stap + self.sam_len])
                well_order = i
                well_stap = self.depth_loc(awmd.iloc[self.loc_li[i]+1])
                well_endp = self.depth_loc(awmd.iloc[self.loc_li[i+1]])
            i+=1
        #print('Time:{:.3f}'.format(b-a))

        #制作单个label, label*mask0=sample
        label_ori=awfea.iloc[stap:endp,:].copy()
        #self.plot_label(label,'label_before_norm')
        ##将对应位置的参考样本选取出来，进而用参考样本的相关数值作均方归一化
        label, np_mean, np_std = self.mean_var_norm(label_ori)
        #self.plot_label(label,'label_after_norm')
        #制作初始的mask0
        mask0=np.where(pd.isnull(label), 0, 1)  #label中null值赋0，nonnulll赋值1  #mask0为numpy
        ##将对应位置的参考样本选取出来，进而用参考样本的相关数值作均方归一化
        label, np_mean, np_std = self.mean_var_norm(label_ori)
        #self.plot_label(label,'label_after_norm')

        label_np=np.where(pd.isnull(label), 0, label)

        return label_np, mask0

    def mean_var_norm(self, label):
        df1_mean=label.mean()
        df1_mean=np.where(pd.isnull(df1_mean), 0, df1_mean)
        #print('df1_mean',df1_mean.shape,df1_mean)
        df1_std=label.std(ddof=0)  #避免std使用index=0的影响
        df1_std=np.where(pd.isnull(df1_std), 0, df1_std)
        #print('df1_std.shape',df1_std.shape, df1_std)

        label_mean_var_norm=(label-df1_mean)/df1_std

        #由于进行gardner比较时，需要用到有实际物理意义的数据，所以这里返回numpy格式的均值和方差
        #直接np.array(df1_mean)时，其shape=(4,) 是一维数组
        np_mean = np.array(df1_mean).reshape((1,len(self.fea_litho)))   #np_mean.shape=(1,4)
        np_std = np.array(df1_std).reshape((1,len(self.fea_litho)))
        return label_mean_var_norm, np_mean, np_std
    def mask_d(self,label,mask0):
        '''制作随机的maskd（需要恢复的），起始点随机'''
        maskd=np.ones((label.shape[0],label.shape[1]))
        a=maskd.shape[0]  #a样本长度，64
        b=maskd.shape[1]  #b特征曲线数目，7

        '''check mask0的哪些列为0，避免让后续的maskd与mask0重合'''
        m0_loc= mask0.sum(axis=0)
        #print('m0_loc',m0_loc)
        loc=np.argwhere(m0_loc==0)
        #print('loc',loc)
        loc = loc.flatten()
        #print('loc',loc)

        ''''用于4条曲线时'''
        if mask0.sum() <= a*(b/2):  #意为mask0多于一半
            maskd=maskd

        else:
            num_maskd=1  #3条或者4条时只给1条maskd
            maskd=self.mask_d_loc(a,b,num_maskd,maskd,loc)

        return maskd

    def mask_d_loc(self,a,b,num_maskd,maskd,loc):
        '''注意是训练还是测试，num选择性注释'''
        num=np.random.randint(b,size=num_maskd)
        '''因为nu列表只有一个参数，所以直接写了num[0]'''
        '''利用set函数判断是否为子集,<=子集;<真子集'''
        while(set(num) <= set(loc) or self.fea_litho.index('GR') in num):
            num=np.random.randint(b,size=num_maskd)
        for i in range(num_maskd):
            maskd[:,num[i]]=0
        return maskd

def custom_collate(batch):
    input_sam = torch.stack([item.x for item in batch])
    label = torch.stack([item.y for item in batch])
    return Data(input_sam=input_sam, label=label)

#实例参数存储对象
os.makedirs(f'/content/drive/MyDrive/logcompletion/loss',exist_ok=True)
from types import SimpleNamespace
args = SimpleNamespace(
    batch_size=800,#4,
    sam_num=2560,
    epochs=11,
    save_model_path=f'/content/drive/MyDrive/logcompletion/model/ctra_exp/',
    lossfigpath=f'/content/drive/MyDrive/logcompletion/figure/ctra_exp/',
    sam_len=512,
    graph_channels=[512,256,128,64,32],
    data_norm_type='',
    LR=0.001,
    fea_litho=['GR', 'RHOB', 'NPHI', 'DTC'],
    need_maskd=True,
    filePath='/content/drive/MyDrive/logcompletion/data/Teamdata/standard_wells/data2/',
    loss_name=f'/content/drive/MyDrive/logcompletion/loss/loss_init.npz',
    filt_size=19,
    drop=0,
    head=1
)

In [ ]:
train_dataset = my_dataset(dfTrain, True, train_maskc, train_loc_li, args.fea_litho, args.sam_num, args.sam_len)
test_dataset = my_dataset(dfTest, True, test_maskc, test_loc_li, args.fea_litho, args.sam_num, args.sam_len)

from torch.utils.data import DataLoader
# 使用标准的 PyTorch DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=args.batch_size,
    num_workers=0,
    shuffle=True,
    drop_last=True,
    collate_fn=custom_collate  # 使用自定义的collate_fn
)
test_loader = DataLoader(
    test_dataset,
    batch_size=args.batch_size,
    num_workers=0,
    shuffle=True,
    drop_last=True,
    collate_fn=custom_collate  # 使用自定义的collate_fn
)

NameError: name 'dfTrain' is not defined

In [ ]:
a = next(iter(test_loader))
a.x.shape,a.y.shape

(torch.Size([800, 4, 512]), torch.Size([800, 4, 512]))

In [ ]:
from functools import reduce
import operator
def count_params(model):
    c = 0
    for p in list(model.parameters()):
        c += reduce(operator.mul,
                    list(p.size()+(2,) if p.is_complex() else p.size()))
    return print(c)

In [ ]:
#convolution in channel features
class conv_obj(nn.Module):
    def __init__(
        self,
        gnn_in_channels: Union[int, Tuple[int, int]],
        gnn_out_channels: int,
        cnn_in_channels: int,
        cnn_out_channels: int,
        batchsize: int = 3,
        down_up: bool = True, #default True == down
        poolstride: int = 2,
        onlypoolout:bool = True,
    ):
        super(conv_obj, self).__init__()
        self.gnn_in_channels = gnn_in_channels
        self.gnn_out_channels = gnn_out_channels
        self.cnn_in_channels = cnn_in_channels
        self.cnn_out_channels = cnn_out_channels
        self.batchsize = batchsize
        self.down_up = down_up
        self.poolstride = poolstride
        self.onlypoolout = onlypoolout


        if isinstance(gnn_in_channels, int):
            gnn_in_channels = (gnn_in_channels, gnn_in_channels)

        self.device = device
        print('device used in conv_obj：',self.device)


        if self.down_up == True:
            self.conv0 =  conv_down_conv_obj(cnn_in_channels,cnn_out_channels,onlypoolout,poolstride).to(self.device)
            self.conv1 =  conv_down_conv_obj(cnn_in_channels,cnn_out_channels,onlypoolout,poolstride).to(self.device)
            self.conv2 =  conv_down_conv_obj(cnn_in_channels,cnn_out_channels,onlypoolout,poolstride).to(self.device)
            self.conv3 =  conv_down_conv_obj(cnn_in_channels,cnn_out_channels,onlypoolout,poolstride).to(self.device)
        else:
            self.conv0 = conv_up_conv_obj(cnn_in_channels,cnn_out_channels,poolstride).to(self.device)
            self.conv1 = conv_up_conv_obj(cnn_in_channels,cnn_out_channels,poolstride).to(self.device)
            self.conv2 = conv_up_conv_obj(cnn_in_channels,cnn_out_channels,poolstride).to(self.device)
            self.conv3 = conv_up_conv_obj(cnn_in_channels,cnn_out_channels,poolstride).to(self.device)

    def forward(self, x):

        CIN,COUT,GIN,GOUT = self.cnn_in_channels,self.cnn_out_channels,self.gnn_in_channels,self.gnn_out_channels
        #print(x.shape)

        y = torch.zeros([self.batchsize,COUT,GOUT]).to(self.device)

        out_cat = torch.zeros([self.batchsize,COUT,GIN]).to(self.device)

        #print(x[0][0:self.batchsize * 4:4][:].shape)
        #print('y',y.shape)
        if self.onlypoolout == True:
            y[0 :self.batchsize * 4:4][:] = self.conv0(x[0:self.batchsize * 4:4][:])
            y[1 :self.batchsize * 4:4][:] = self.conv1(x[1:self.batchsize * 4:4][:])
            y[2 :self.batchsize * 4:4][:] = self.conv2(x[2:self.batchsize * 4:4][:])
            y[3 :self.batchsize * 4:4][:] = self.conv3(x[3:self.batchsize * 4:4][:])
        else:
            #temp = self.conv0(x[0][0:self.batchsize * 4:4][:])
            #print('temp length ',len(temp))
            #print('temp ',temp[0].shape)
            y[0 :self.batchsize * 4:4][:], out_cat[0 :self.batchsize * 4:4][:] = self.conv0(x[0:self.batchsize * 4:4][:])
            y[1 :self.batchsize * 4:4][:], out_cat[1 :self.batchsize * 4:4][:] = self.conv1(x[1:self.batchsize * 4:4][:])
            y[2 :self.batchsize * 4:4][:], out_cat[2 :self.batchsize * 4:4][:] = self.conv2(x[2:self.batchsize * 4:4][:])
            y[3 :self.batchsize * 4:4][:], out_cat[3 :self.batchsize * 4:4][:] = self.conv3(x[3:self.batchsize * 4:4][:])

        return y, out_cat

#用于conv_obj的下采样卷积模块
class conv_down_conv_obj(nn.Module):
    def __init__(self,in_ch,out_ch,onlypoolout,poolstride = 2):
        super(conv_down_conv_obj, self).__init__()
        self.onlypoolout = onlypoolout
        self.Conv = nn.Sequential(
            nn.Conv1d(in_channels=in_ch,out_channels=out_ch,kernel_size=7,stride=1,padding=3),
            nn.BatchNorm1d(out_ch),
            nn.LeakyReLU(negative_slope=0.2),
            #nn.ELU(),

            nn.Conv1d(in_channels=out_ch,out_channels=out_ch,kernel_size=7,stride=1,padding=3),
            nn.BatchNorm1d(out_ch),
            nn.LeakyReLU(negative_slope=0.2),
            #nn.ELU()
        )

        self.downsample= nn.Sequential(
            nn.MaxPool1d(5, stride=poolstride,padding = 2)
        )


    def forward(self,x):

        out_cat = self.Conv(x)
        out=self.downsample(out_cat)
        #print('out.size()',out.size(),'out.dtype',out.dtype,'out.type()',out.type())

        if self.onlypoolout==True:
            return out
        else:
            return out, out_cat

#用于conv_obj的上采样卷积模块
class conv_up_conv_obj(nn.Module):
    def __init__(self,in_ch,out_ch,poolstride = 2):
        super(conv_up_conv_obj, self).__init__()

        self.layers = nn.Sequential(
            nn.Conv1d(in_channels = in_ch, out_channels = 2*out_ch, kernel_size=7, stride=1, padding=3),
            nn.BatchNorm1d(2*out_ch),
            nn.LeakyReLU(negative_slope=0.2),
            #nn.ELU(),

            nn.Conv1d(in_channels = 2*out_ch, out_channels = 2*out_ch, kernel_size=7, stride=1, padding=3),
            nn.BatchNorm1d(2*out_ch),
            nn.LeakyReLU(negative_slope=0.2),

            nn.ConvTranspose1d(in_channels=2*out_ch,out_channels=out_ch,kernel_size=7,stride=poolstride,padding=3,output_padding=(poolstride-1)),
            nn.BatchNorm1d(out_ch),
            nn.LeakyReLU(negative_slope=0.2),
            #nn.ELU(),
        )

    def forward(self,x):
        out = self.layers(x)

        return out

#Unet结构上下采样部分的中间衔接（上采样）
class upsample(nn.Module):
    '''仅进行上采样（将上采样与和聚合模块分割开）'''
    def __init__(self,gnn_out_ch, cnn_in_ch, cnn_out_ch, bsize, poolstride=2):
        super(upsample,self).__init__()
        self.bsize = bsize
        self.cout = cnn_out_ch
        self.gout = gnn_out_ch

        '''GNN_conv_upsample与upsample基于相同的基类(nn.Module)，所以在外面将upsample进行.to(device)后，这里不需要将GNN_conv_upsample再.to(device)'''
        self.conv1 = conv_up_conv_obj(cnn_in_ch,cnn_out_ch,poolstride)
        self.conv0 = conv_up_conv_obj(cnn_in_ch,cnn_out_ch,poolstride)
        self.conv2 = conv_up_conv_obj(cnn_in_ch,cnn_out_ch,poolstride)
        self.conv3 = conv_up_conv_obj(cnn_in_ch,cnn_out_ch,poolstride)

    def forward(self,x):
        y = torch.zeros([self.bsize, self.cout, self.gout]).to(device)

        y[0 :self.bsize * 4:4][:] = self.conv0(x[0:self.bsize * 4:4][:])
        y[1 :self.bsize * 4:4][:] = self.conv1(x[1:self.bsize * 4:4][:])
        y[2 :self.bsize * 4:4][:] = self.conv2(x[2:self.bsize * 4:4][:])
        y[3 :self.bsize * 4:4][:] = self.conv3(x[3:self.bsize * 4:4][:])

        return y

#Unet结构下采样部分中，补偿torch.cat导致的通道数翻倍
class cat_fuse(nn.Module):
    """
    卷积模块，减半通道数。
    """
    def __init__(self,cnn_in_ch,cnn_out_ch):
        super(cat_fuse,self).__init__()

        self.Conv = nn.Sequential(
            nn.Conv1d(in_channels=cnn_in_ch,out_channels=cnn_out_ch,kernel_size=7,stride=1,padding=3),
            nn.BatchNorm1d(cnn_out_ch),
            nn.LeakyReLU(negative_slope=0.2),
        )
    def forward(self,x):
        y = self.Conv(x)

        return y

#mlp用于学习mean与std
class mean_std_head(torch.nn.Module):
    def __init__(self):
        super(mean_std_head, self).__init__()

        self.mlp1 = MLP(4*512, 4*128, 4*32)
        self.mlp2 = MLP(4*32, 4*8, 8)

        #self.dropout = nn.Dropout(p=0.5)


    def forward(self, x, batch):

        y = torch.stack(unbatch(x, batch))
        y = torch.flatten(y,1,-1)

        y = self.mlp1(y)
        #y = self.dropout(y)
        y = self.mlp2(y)
        #y = self.dropout(y)
        #print('全连接后y', y.size(), y)

        return y


class MLP(torch.nn.Module):
    def __init__(self, in_ch, hid_ch, out_ch):
        super(MLP, self).__init__()

        self.mlp = nn.Sequential(
            nn.Linear(in_ch, hid_ch),
            nn.LeakyReLU(negative_slope=0.2),

            nn.Linear(hid_ch, hid_ch),
            nn.LeakyReLU(negative_slope=0.2),

            nn.Linear(hid_ch, out_ch),
        )

    def forward(self, x):
        x = self.mlp(x)

        return x

# Fourier Feature Extractor core
class SpectralConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1):
        super(SpectralConv1d, self).__init__()

        """
        3D Fourier layer. It does FFT, linear transform, and Inverse FFT.
        """

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1 #Number of Fourier modes to multiply, at most floor(N/2) + 1


        self.scale = (1 / (in_channels * out_channels))
        self.weights1 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, dtype=torch.cfloat))
        self.weights3 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, dtype=torch.cfloat))
        self.weights4 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, dtype=torch.cfloat))

    # Complex multiplication
    def compl_mul1d(self, input, weights):
        # (batch, in_channel, x), (in_channel, out_channel, x) -> (batch, out_channel, x)
        return torch.einsum("bix,iox->box", input, weights)

    def forward(self, x):
        batchsize = x.shape[0]
        #Compute Fourier coeffcients up to factor of e^(- something constant)
        x_ft = torch.fft.rfftn(x, dim=[-1]).to(x.device)

        # Multiply relevant Fourier modes
        out_ft = torch.zeros(batchsize, self.out_channels, x.size(-1)//2 + 1, dtype=torch.cfloat, device=x.device)
        out_ft[:, :, :self.modes1] = \
            self.compl_mul1d(x_ft[:, :, :self.modes1], self.weights1.to(x.device))
        out_ft[:, :, -self.modes1:] = \
            self.compl_mul1d(x_ft[:, :, -self.modes1:], self.weights2.to(x.device))
        out_ft[:, :, :self.modes1] = \
            self.compl_mul1d(x_ft[:, :, :self.modes1], self.weights3.to(x.device))
        out_ft[:, :, -self.modes1:] = \
            self.compl_mul1d(x_ft[:, :, -self.modes1:], self.weights4.to(x.device))

        #Return to physical space
        x = torch.fft.irfftn(out_ft, s=(x.size(-1)))
        return x

class MLP_(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels):
        super(MLP_, self).__init__()
        self.mlp1 = nn.Conv1d(in_channels, mid_channels, 1)
        self.mlp2 = nn.Conv1d(mid_channels, out_channels, 1)

    def forward(self, x):
        x = self.mlp1(x)
        x = F.gelu(x)
        x = self.mlp2(x)
        return x

class FourierFeatureExtractor(nn.Module):
    def __init__(self, in_channels=4, out_channels=1, mid_channels=8, modes1=8):
        super(FourierFeatureExtractor,self).__init__()
        self.fconv1 = SpectralConv1d(in_channels, mid_channels, modes1)
        self.fconv2 = SpectralConv1d(mid_channels, mid_channels, modes1)
        self.fconv3 = SpectralConv1d(mid_channels, mid_channels, modes1)
        self.mlp1 = MLP_(mid_channels, mid_channels, mid_channels)
        self.mlp2 = MLP_(mid_channels, mid_channels, mid_channels)
        self.mlp3 = MLP_(mid_channels, mid_channels, mid_channels)
        self.w1 = nn.Conv1d(in_channels, mid_channels, 1)
        self.w2 = nn.Conv1d(mid_channels, mid_channels, 1)
        self.w3 = nn.Conv1d(mid_channels, mid_channels, 1)
        self.out_proj = MLP_(mid_channels, out_channels, mid_channels)


    def forward(self,x):
        # x (32,4,512)
        # f-1
        x_f = self.fconv1(x)
        x_f = self.mlp1(x_f)

        # c-1
        x_c =self.w1(x)

        x = x_f + x_c
        x = F.gelu(x)
        # f-2
        x_f = self.fconv2(x)
        x_f = self.mlp2(x_f)

        # c-2
        x_c =self.w2(x)
        x = x_f + x_c
        x = F.gelu(x)

        # f-3
        x_f = self.fconv3(x)
        x_f = self.mlp3(x_f)

        # c-3
        x_c =self.w3(x)
        x = x_f + x_c
        x = F.gelu(x)

        x = self.out_proj(x)
        return x


# U-Net
#meta model to be trained
class FNPG(torch.nn.Module):

    def __init__(self,gnn_ch,batchsize,fea_litho,head,drop):
        super(FNPG, self).__init__()
        """
        input:(batch_size,num_curves,channels,sam_len)
        output:(batch_size,num_curves,sam_len)
        """
        CNN_ch = [4,32,64,128,256,512] #gnn_ch = [512,256,128,64,32]

        result_ch=4

        self.bsize = batchsize
        self.fea_n = len(fea_litho)

        #self.head = mean_std_head()#到底还用不用神经网络去反标准化了？
        self.gd0 = conv_obj(gnn_ch[0],gnn_ch[1],CNN_ch[0],CNN_ch[1],batchsize = batchsize,onlypoolout=False,down_up = True)
        self.gd1 = conv_obj(gnn_ch[1],gnn_ch[2],CNN_ch[1],CNN_ch[2],batchsize = batchsize,onlypoolout=False,down_up = True)
        self.gd2 = conv_obj(gnn_ch[2],gnn_ch[3],CNN_ch[2],CNN_ch[3],batchsize = batchsize,onlypoolout=False,down_up = True)

        self.FFE_up1 = FourierFeatureExtractor(CNN_ch[1],CNN_ch[1], mid_channels=32, modes1=16)
        self.FFE_up2 = FourierFeatureExtractor(CNN_ch[2],CNN_ch[2], mid_channels=32, modes1=16)
        self.FFE_up3 = FourierFeatureExtractor(CNN_ch[3],CNN_ch[3], mid_channels=32, modes1=16)


        self.u0 = upsample(gnn_ch[2],CNN_ch[3],CNN_ch[3], bsize=batchsize, poolstride=2).to(device)
        self.cat0 =cat_fuse(CNN_ch[4],CNN_ch[3]).to(device)

        self.u1 = upsample(gnn_ch[1],CNN_ch[3],CNN_ch[2], bsize=batchsize, poolstride=2).to(device)
        self.cat1 =cat_fuse(CNN_ch[3],CNN_ch[2]).to(device)

        self.u2 = upsample(gnn_ch[0],CNN_ch[2],CNN_ch[1], bsize=batchsize, poolstride=2).to(device)
        self.cat2 =cat_fuse(CNN_ch[2],CNN_ch[1]).to(device)

        self.FFE_down1 = FourierFeatureExtractor(CNN_ch[1],CNN_ch[1], mid_channels=32, modes1=16)
        self.FFE_down2 = FourierFeatureExtractor(CNN_ch[2],CNN_ch[2], mid_channels=32, modes1=16)
        self.FFE_down3 = FourierFeatureExtractor(CNN_ch[3],CNN_ch[3], mid_channels=32, modes1=16)


        self.result = nn.Sequential(
            nn.Conv1d(CNN_ch[1],CNN_ch[1],kernel_size=7,stride=1,padding=3),
            nn.BatchNorm1d(CNN_ch[1]),
            nn.LeakyReLU(negative_slope=0.2),

            nn.Conv1d(CNN_ch[1],CNN_ch[1],kernel_size=7,stride=1,padding=3),
            nn.BatchNorm1d(CNN_ch[1]),
            nn.LeakyReLU(negative_slope=0.2),

            nn.Conv1d(CNN_ch[1],result_ch,kernel_size=7,stride=1,padding=3),
        )

    def forward(self,data):
        x = data.x

        out, cat0 = self.gd0(x)

        out = self.FFE_up1(out)

        out, cat1 = self.gd1(out)

        out = self.FFE_up2(out)

        out, cat2 = self.gd2(out)

        out = self.FFE_up3(out)

        out = self.u0(out)

        out = torch.cat((cat2,out),dim=1)

        out = self.cat0(out)

        out = self.FFE_down3(out)

        out = self.u1(out)

        out = torch.cat((cat1,out),dim=1)

        out = self.cat1(out)
        out = self.FFE_down2(out)

        out = self.u2(out)

        out = torch.cat((cat0,out),dim=1)

        out = self.cat2(out)
        out = self.FFE_down1(out)


        out = self.result(out)
        out = out.squeeze(1)

        return out
#
class FNPG_apple(torch.nn.Module):

    def __init__(self,gnn_ch,batchsize,fea_litho,head,drop):
        super(FNPG_apple, self).__init__()
        """
        input:(batch_size,num_curves,channels,sam_len)
        output:(batch_size,num_curves,sam_len)
        """
        CNN_ch = [4,32,64,128,256,512] #gnn_ch = [512,256,128,64,32]

        result_ch=5

        self.bsize = batchsize
        self.fea_n = len(fea_litho)

        #self.head = mean_std_head()#到底还用不用神经网络去反标准化了？
        self.gd0 = conv_obj(gnn_ch[0],gnn_ch[1],CNN_ch[0],CNN_ch[1],batchsize = batchsize,onlypoolout=False,down_up = True)
        self.gd1 = conv_obj(gnn_ch[1],gnn_ch[2],CNN_ch[1],CNN_ch[2],batchsize = batchsize,onlypoolout=False,down_up = True)
        self.gd2 = conv_obj(gnn_ch[2],gnn_ch[3],CNN_ch[2],CNN_ch[3],batchsize = batchsize,onlypoolout=False,down_up = True)

        self.FFE_up1 = FourierFeatureExtractor(CNN_ch[1],CNN_ch[1], mid_channels=32, modes1=16)
        self.FFE_up2 = FourierFeatureExtractor(CNN_ch[2],CNN_ch[2], mid_channels=32, modes1=16)
        self.FFE_up3 = FourierFeatureExtractor(CNN_ch[3],CNN_ch[3], mid_channels=32, modes1=16)


        self.u0 = upsample(gnn_ch[2],CNN_ch[3],CNN_ch[3], bsize=batchsize, poolstride=2).to(device)
        self.cat0 =cat_fuse(CNN_ch[4],CNN_ch[3]).to(device)

        self.u1 = upsample(gnn_ch[1],CNN_ch[3],CNN_ch[2], bsize=batchsize, poolstride=2).to(device)
        self.cat1 =cat_fuse(CNN_ch[3],CNN_ch[2]).to(device)

        self.u2 = upsample(gnn_ch[0],CNN_ch[2],CNN_ch[1], bsize=batchsize, poolstride=2).to(device)
        self.cat2 =cat_fuse(CNN_ch[2],CNN_ch[1]).to(device)

        self.FFE_down1 = FourierFeatureExtractor(CNN_ch[1],CNN_ch[1], mid_channels=32, modes1=16)
        self.FFE_down2 = FourierFeatureExtractor(CNN_ch[2],CNN_ch[2], mid_channels=32, modes1=16)
        self.FFE_down3 = FourierFeatureExtractor(CNN_ch[3],CNN_ch[3], mid_channels=32, modes1=16)


        self.result = nn.Sequential(
            nn.Conv1d(CNN_ch[1],CNN_ch[1],kernel_size=7,stride=1,padding=3),
            nn.BatchNorm1d(CNN_ch[1]),
            nn.LeakyReLU(negative_slope=0.2),

            nn.Conv1d(CNN_ch[1],CNN_ch[1],kernel_size=7,stride=1,padding=3),
            nn.BatchNorm1d(CNN_ch[1]),
            nn.LeakyReLU(negative_slope=0.2),

            nn.Conv1d(CNN_ch[1],result_ch,kernel_size=7,stride=1,padding=3),
        )

    def forward(self,data):
        x = data.x

        out, cat0 = self.gd0(x)

        out = self.FFE_up1(out) + out

        out, cat1 = self.gd1(out)

        out = self.FFE_up2(out) + out

        out, cat2 = self.gd2(out)

        out = self.FFE_up3(out) + out

        out = self.u0(out)

        out = torch.cat((cat2,out),dim=1)

        out = self.cat0(out)

        out = self.FFE_down3(out) + out

        out = self.u1(out)

        out = torch.cat((cat1,out),dim=1)

        out = self.cat1(out)
        out = self.FFE_down2(out) + out

        out = self.u2(out)

        out = torch.cat((cat0,out),dim=1)

        out = self.cat2(out)
        out = self.FFE_down1(out) + out


        out = self.result(out)
        out = out.squeeze(1)

        return out

class Unet_fused(torch.nn.Module):

    def __init__(self,gnn_ch,batchsize,fea_litho,head,drop):
        super(Unet_fused, self).__init__()
        """
        input:(batch_size,num_curves,sam_len)
        output:(batch_size,result_ch,sam_len)
        """
        CNN_ch = [4,32,64,128,256,512] #gnn_ch = [512,256,128,64,32]

        result_ch=4

        self.bsize = batchsize
        self.fea_n = len(fea_litho)

        self.head = mean_std_head()
        self.gd0 = conv_obj(gnn_ch[0],gnn_ch[1],CNN_ch[0],CNN_ch[1],batchsize = batchsize,onlypoolout=False,down_up = True)
        self.gd1 = conv_obj(gnn_ch[1],gnn_ch[2],CNN_ch[1],CNN_ch[2],batchsize = batchsize,onlypoolout=False,down_up = True)
        self.gd2 = conv_obj(gnn_ch[2],gnn_ch[3],CNN_ch[2],CNN_ch[3],batchsize = batchsize,onlypoolout=False,down_up = True)

        self.FFE_up1 = FourierFeatureExtractor(CNN_ch[1],CNN_ch[1], mid_channels=32, modes1=16)
        self.FFE_up2 = FourierFeatureExtractor(CNN_ch[2],CNN_ch[2], mid_channels=32, modes1=16)
        self.FFE_up3 = FourierFeatureExtractor(CNN_ch[3],CNN_ch[3], mid_channels=32, modes1=16)


        self.u0 = upsample(gnn_ch[2],CNN_ch[3],CNN_ch[3], bsize=batchsize, poolstride=2).to(device)
        self.cat0 =cat_fuse(CNN_ch[4],CNN_ch[3]).to(device)

        self.u1 = upsample(gnn_ch[1],CNN_ch[3],CNN_ch[2], bsize=batchsize, poolstride=2).to(device)
        self.cat1 =cat_fuse(CNN_ch[3],CNN_ch[2]).to(device)

        self.u2 = upsample(gnn_ch[0],CNN_ch[2],CNN_ch[1], bsize=batchsize, poolstride=2).to(device)
        self.cat2 =cat_fuse(CNN_ch[2],CNN_ch[1]).to(device)


        self.result = nn.Sequential(
            nn.Conv1d(CNN_ch[1],CNN_ch[1],kernel_size=7,stride=1,padding=3),
            nn.BatchNorm1d(CNN_ch[1]),
            nn.LeakyReLU(negative_slope=0.2),

            nn.Conv1d(CNN_ch[1],CNN_ch[1],kernel_size=7,stride=1,padding=3),
            nn.BatchNorm1d(CNN_ch[1]),
            nn.LeakyReLU(negative_slope=0.2),

            nn.Conv1d(CNN_ch[1],result_ch,kernel_size=7,stride=1,padding=3),
        )

    def forward(self,data):
        x = data.x

        out, cat0 = self.gd0(x)


        out, cat1 = self.gd1(out)


        out, cat2 = self.gd2(out)


        out = self.u0(out)


        out = torch.cat((cat2,out),dim=1)

        out = self.cat0(out)



        #out = self.gu0(out,edge_index_ini, edge_ini_list)


        out = self.u1(out)

        out = torch.cat((cat1,out),dim=1)

        out = self.cat1(out)


        #out = self.gu1(out,edge_index, edge_list)


        out = self.u2(out)

        out = torch.cat((cat0,out),dim=1)

        out = self.cat2(out)


        #out =self.gu2(out,edge_index, edge_list)

        out = self.result(out)



        out = out.squeeze(1)

        return out


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self):
        super(UNet, self).__init__()

        # Encoder (Downsampling)
        self.enc1 = DoubleConv(4, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)
        self.enc5 = DoubleConv(512, 1024)

        # Decoder (Upsampling)
        self.dec4 = DoubleConv(1024 + 512, 512)
        self.dec3 = DoubleConv(512 + 256, 256)
        self.dec2 = DoubleConv(256 + 128, 128)
        self.dec1 = DoubleConv(128 + 64, 64)

        self.pool = nn.MaxPool1d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='linear', align_corners=True)

        self.final_conv = nn.Conv1d(64, 5, kernel_size=1)

    def forward(self, x):
        x = x.x
        # Encoder
        enc1 = self.enc1(x)
        x = self.pool(enc1)
        enc2 = self.enc2(x)
        x = self.pool(enc2)
        enc3 = self.enc3(x)
        x = self.pool(enc3)
        enc4 = self.enc4(x)
        x = self.pool(enc4)

        # Bridge
        x = self.enc5(x)

        # Decoder
        x = self.upsample(x)
        x = torch.cat([x, enc4], dim=1)
        x = self.dec4(x)

        x = self.upsample(x)
        x = torch.cat([x, enc3], dim=1)
        x = self.dec3(x)

        x = self.upsample(x)
        x = torch.cat([x, enc2], dim=1)
        x = self.dec2(x)

        x = self.upsample(x)
        x = torch.cat([x, enc1], dim=1)
        x = self.dec1(x)

        return self.final_conv(x)



In [ ]:
from types import SimpleNamespace

# 因为要做4预测7的实验，路径用老的{STAGE}检索不合适，附一个的吧
STAGE = 'Reconstruction'
os.makedirs(f'/content/drive/MyDrive/logcompletion/model/ctra_exp_{STAGE}/',exist_ok=True)
os.makedirs(f'/content/drive/MyDrive/logcompletion/figure/ctra_exp_{STAGE}/',exist_ok=True)
os.makedirs(f'/content/drive/MyDrive/logcompletion/loss_{STAGE}/',exist_ok=True)
args = SimpleNamespace(
    batch_size=800,#4
    sam_num=2560,
    epochs=1,
    save_model_path=f'/content/drive/MyDrive/logcompletion/model/ctra_exp_{STAGE}/',
    lossfigpath=f'/content/drive/MyDrive/logcompletion/figure/ctra_exp_{STAGE}/',
    sam_len=512,
    graph_channels=[512,256,128,64,32],
    data_norm_type='',
    LR=0.001,
    #['RHOB', 'NPHI', 'DTC', 'RDEP', 'SP'，'BS']
    fea_litho=['GR', 'RHOB', 'NPHI', 'DTC'],
    need_maskd=True,
    filePath='/content/drive/MyDrive/logcompletion/data/Teamdata/standard_wells/data2/',
    loss_name=f'/content/drive/MyDrive/logcompletion/loss_{STAGE}/loss_init.npz',
    filt_size=19,
    drop=0,
    head=1
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
a = next(iter(train_loader))
a.x = a.x.view(args.batch_size,len(args.fea_litho),args.sam_len)
a.y = a.y.view(args.batch_size,-1,args.sam_len)
a.x.shape,a.y.shape

(torch.Size([800, 4, 512]), torch.Size([800, 4, 512]))

In [ ]:
# 检查drop_last
with torch.no_grad():
    for a in tqdm(test_loader):
        a = a.to(device)
        print(a.x.shape)
        a.x = a.x.view(args.batch_size,len(args.fea_litho),args.sam_len)

 25%|██▌       | 1/4 [00:02<00:06,  2.23s/it]

torch.Size([600, 4, 512])


 50%|█████     | 2/4 [00:04<00:04,  2.08s/it]

torch.Size([600, 4, 512])


 75%|███████▌  | 3/4 [00:06<00:02,  2.02s/it]

torch.Size([600, 4, 512])


100%|██████████| 4/4 [00:08<00:00,  2.03s/it]

torch.Size([600, 4, 512])


In [ ]:
class CombinedLoss(nn.Module):
    def __init__(self, window_size=5, size_average=True, alpha=0.5):
        super(CombinedLoss, self).__init__()
        self.window_size = window_size
        self.size_average = size_average
        self.channel = 1
        self.window = self.create_window(window_size)

        self.alpha = alpha  # 用于平衡两种损失的权重

    def create_window(self, window_size):
        # 创建一个高斯窗口
        window = torch.Tensor([exp(-(x - window_size//2)**2/float(2*1.5**2)) for x in range(window_size)])
        return window.unsqueeze(0)#1

    def forward(self, img1, img2):
        """
        img1:y_pred
        img2:y_true
        """
        (_, channel, length) = img1.size()

        if channel == self.channel and self.window.data.type() == img1.data.type():
            window = self.window
        else:
            window = self.create_window(self.window_size).to(img1.device).type_as(img1)
            self.window = window
            self.channel = channel

        #均方误差和相关系数误差
        # 确保输入形状正确
        y_pred = img1
        y_true = img2
        assert y_pred.shape == y_true.shape
        assert y_pred.shape[1] == 1  # 确保中间维度为1

        batch_size = y_pred.shape[0]
        seq_len = y_pred.shape[2]

        # 重塑张量以便于计算
        y_pred_flat = y_pred.view(batch_size, seq_len)
        y_true_flat = y_true.view(batch_size, seq_len)

        # 计算相关系数损失
        y_pred_mean = y_pred_flat.mean(dim=1, keepdim=True)
        y_true_mean = y_true_flat.mean(dim=1, keepdim=True)

        covariance = ((y_pred_flat - y_pred_mean) * (y_true_flat - y_true_mean)).sum(dim=1)

        y_pred_std = torch.sqrt(((y_pred_flat - y_pred_mean) ** 2).sum(dim=1))
        y_true_std = torch.sqrt(((y_true_flat - y_true_mean) ** 2).sum(dim=1))

        correlation = covariance / (y_pred_std * y_true_std + 1e-8)
        corr_loss = 1 - correlation.mean()

        # 计算均方误差损失
        mse_loss = F.mse_loss(y_pred, y_true)
        """
        # Unpack input data
        gr, rhob, nphi, dtc = input_data.unbind(dim=1)
        nphi = nphi.view(nphi.size(0),1,nphi.size(1))
        assert y_pred.shape == nphi.shape
        archie_loss = F.mse_loss(y_pred, 1/nphi)
        """
        # 组合损失
        combined_loss = self.alpha * corr_loss + (1 - self.alpha) * mse_loss

        return combined_loss #+ self._ssim(img1, img2, window, self.window_size, channel)
    def _ssim(self, img1, img2, window, window_size, channel):
        window = window.unsqueeze(0) #add a dimension to the window tensor
        #print(img1.shape,window.shape)#torch.Size([4, 1, 512]) torch.Size([1, 5, 1])
        #(1,1,5)
        mu1 = F.conv1d(img1, window, padding=0, stride=5, groups=channel)
        mu2 = F.conv1d(img2, window, padding=0, stride=5, groups=channel)

        mu1_sq = mu1.pow(2)
        mu2_sq = mu2.pow(2)
        mu1_mu2 = mu1 * mu2

        sigma1_sq = F.conv1d(img1 * img1, window, padding=0, stride=5, groups=channel) - mu1_sq
        sigma2_sq = F.conv1d(img2 * img2, window, padding=0, stride=5, groups=channel) - mu2_sq
        sigma12 = F.conv1d(img1 * img2, window, padding=0, stride=5, groups=channel) - mu1_mu2

        C1 = 0.01**2
        C2 = 0.03**2

        ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))

        if self.size_average:
            return 1 - ssim_map.mean()
        else:
            return 1 - ssim_map.mean(1).mean(1)

class CustomLossWithDerivativePenalty(nn.Module):
    def __init__(self, alpha=0.1, beta=0.01):
        super().__init__()
        self.alpha = alpha  # 一阶导数的权重
        self.beta = beta    # 二阶导数的权重
        self.mse = nn.MSELoss()

    def forward(self, y_pred, y_true):
        # 基本MSE损失
        mse_loss = self.mse(y_pred, y_true)

        # 计算一阶导数（使用中心差分法）
        dy = (y_pred[:, :, 2:] - y_pred[:, :, :-2]) / 2

        # 计算二阶导数
        d2y = y_pred[:, :, 2:] - 2 * y_pred[:, :, 1:-1] + y_pred[:, :, :-2]

        # 计算导数惩罚
        derivative_penalty = -self.alpha * torch.mean(torch.abs(dy)) + self.beta * torch.mean(torch.square(d2y))

        # 总损失
        total_loss = mse_loss #+ derivative_penalty

        return total_loss

class CustomLossWithAdaptiveDerivativePenalty(nn.Module):
    """
    自定义损失函数，结合均方误差、导数惩罚和真实值导数的MSE。
    """
    def __init__(self, sequence_length=512, init_alpha=0.1, init_beta=0.01):
        super().__init__()
        self.sequence_length = sequence_length
        self.mse = nn.MSELoss()

        # 将 alpha 和 beta 设置为可学习的参数
        self.log_alpha = nn.Parameter(torch.log(torch.tensor(init_alpha)))
        self.log_beta = nn.Parameter(torch.log(torch.tensor(init_beta)))

    def forward(self, y_pred, y_true):
        # 基本MSE损失
        mse_loss = self.mse(y_pred, y_true)

        # 计算 y_pred 的一阶导数（使用中心差分法，考虑边界）
        dy_pred = torch.zeros_like(y_pred)
        dy_pred[:, :, 1:-1] = (y_pred[:, :, 2:] - y_pred[:, :, :-2]) / 2
        dy_pred[:, :, 0] = y_pred[:, :, 1] - y_pred[:, :, 0]  # 前边界
        dy_pred[:, :, -1] = y_pred[:, :, -1] - y_pred[:, :, -2]  # 后边界

        # 计算 y_true 的一阶导数（使用中心差分法，考虑边界）
        dy_true = torch.zeros_like(y_true)
        dy_true[:, :, 1:-1] = (y_true[:, :, 2:] - y_true[:, :, :-2]) / 2
        dy_true[:, :, 0] = y_true[:, :, 1] - y_true[:, :, 0]  # 前边界
        dy_true[:, :, -1] = y_true[:, :, -1] - y_true[:, :, -2]  # 后边界

        # 计算一阶导数的MSE损失
        derivative_mse_loss = self.mse(dy_pred, dy_true)

        # 计算二阶导数（考虑边界）
        d2y = torch.zeros_like(y_pred)
        d2y[:, :, 1:-1] = y_pred[:, :, 2:] - 2 * y_pred[:, :, 1:-1] + y_pred[:, :, :-2]
        d2y[:, :, 0] = y_pred[:, :, 2] - 2 * y_pred[:, :, 1] + y_pred[:, :, 0]
        d2y[:, :, -1] = y_pred[:, :, -1] - 2 * y_pred[:, :, -2] + y_pred[:, :, -3]

        # 使用 exp 来确保 alpha 和 beta 始终为正
        alpha = torch.exp(self.log_alpha)
        beta = torch.exp(self.log_beta)

        # 计算导数惩罚
        derivative_penalty = -alpha * torch.mean(torch.abs(dy_pred)) + beta * torch.mean(torch.square(d2y))
        #print(mse_loss.shape, derivative_mse_loss.shape, derivative_penalty.shape)
        # 总损失
        total_loss = mse_loss + 1 * derivative_mse_loss + 1 * derivative_penalty

        return total_loss

    def get_alpha_beta(self):
        return torch.exp(self.log_alpha).item(), torch.exp(self.log_beta).item()

import pywt
class CustomLossWithAdaptiveDerivativePenaltyAndWavelet(nn.Module):
    def __init__(self, sequence_length=512, init_alpha=0.1, init_beta=0.01, init_gamma=0.1, wavelet='db4', mode='zero', level=None):
        super().__init__()
        self.sequence_length = sequence_length
        self.mse = nn.MSELoss()
        self.mse_none = nn.MSELoss(reduction='none')


        self.log_alpha = nn.Parameter(torch.log(torch.tensor(init_alpha)))
        self.log_beta = nn.Parameter(torch.log(torch.tensor(init_beta)))
        self.log_gamma = nn.Parameter(torch.log(torch.tensor(init_gamma)))

        self.wavelet = wavelet
        self.mode = mode
        self.level = level

    def wavelet_transform(self, x):
        # 对输入进行小波变换并重构为原始长度
        coeffs = []
        for i in range(x.shape[0]):
            for j in range(x.shape[1]):
                # 进行小波分解
                coeff = pywt.wavedec(x[i, j].cpu().numpy(), self.wavelet, mode=self.mode, level=self.level)

                # 保留近似系数，将细节系数置零
                new_coeffs = [coeff[0]] + [np.zeros_like(c) for c in coeff[1:]]

                # 重构信号
                reconstructed = pywt.waverec(new_coeffs, self.wavelet, mode=self.mode)

                # 确保重构信号长度与原始信号相同
                if len(reconstructed) > self.sequence_length:
                    reconstructed = reconstructed[:self.sequence_length]
                elif len(reconstructed) < self.sequence_length:
                    reconstructed = np.pad(reconstructed, (0, self.sequence_length - len(reconstructed)))

                coeffs.append(torch.tensor(reconstructed).to(x.device))

        return torch.stack(coeffs).view(x.shape)

    def forward(self, y_pred, y_true):
        # 基本MSE损失
        mse_loss = F.mse_loss(y_pred, y_true)

        # 计算 y_pred 的一阶导数
        dy_pred = torch.zeros_like(y_pred)
        dy_pred[:, :, 1:-1] = (y_pred[:, :, 2:] - y_pred[:, :, :-2]) / 2
        dy_pred[:, :, 0] = y_pred[:, :, 1] - y_pred[:, :, 0]
        dy_pred[:, :, -1] = y_pred[:, :, -1] - y_pred[:, :, -2]

        # 计算 y_true 的一阶导数
        dy_true = torch.zeros_like(y_true)
        dy_true[:, :, 1:-1] = (y_true[:, :, 2:] - y_true[:, :, :-2]) / 2
        dy_true[:, :, 0] = y_true[:, :, 1] - y_true[:, :, 0]
        dy_true[:, :, -1] = y_true[:, :, -1] - y_true[:, :, -2]

        # 计算一阶导数的MSE损失
        derivative_mse_loss = self.mse_none(dy_pred, dy_true)
        # 计算二阶导数
        d2y = torch.zeros_like(y_pred)
        d2y[:, :, 1:-1] = y_pred[:, :, 2:] - 2 * y_pred[:, :, 1:-1] + y_pred[:, :, :-2]
        d2y[:, :, 0] = y_pred[:, :, 2] - 2 * y_pred[:, :, 1] + y_pred[:, :, 0]
        d2y[:, :, -1] = y_pred[:, :, -1] - 2 * y_pred[:, :, -2] + y_pred[:, :, -3]

        alpha = torch.exp(self.log_alpha)
        beta = torch.exp(self.log_beta)
        gamma = torch.exp(self.log_gamma)

        # 计算导数惩罚
        derivative_penalty = -alpha * torch.mean(torch.abs(dy_pred)) + beta * torch.mean(torch.square(d2y))
        # 计算小波系数并用作权重
        wavelet_coeffs = self.wavelet_transform(y_true)
        weights = torch.abs(wavelet_coeffs)
        weights = weights / (weights.sum(dim=-1, keepdim=True) + 1e-8)
        #print('weights',weights.shape)
        #print('mse_loss',mse_loss.shape)


        # 应用权重到MSE损失
        weighted_mse_loss = (mse_loss * (1 + gamma * weights)).mean()
        weighted_derivative_mse_loss = (derivative_mse_loss * (1 + weights)).mean() #alpha之前是造成负值loss的derivative_penalty中的一个系数

        # 总损失
        total_loss = weighted_mse_loss + weighted_derivative_mse_loss + derivative_penalty
        #total_loss = F.relu(total_loss)
        return total_loss


    def get_alpha_beta_gamma(self):
        return torch.exp(self.log_alpha).item(), torch.exp(self.log_beta).item(), torch.exp(self.log_gamma).item()

In [ ]:
# 神经网络最佳模型保存与加载
def save_checkpoint(model, optimizer, epoch, best_val_loss, path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_val_loss': best_val_loss,
    }, path)

def load_best_model(model, optimizer, path):
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    return model, optimizer, checkpoint['epoch'], checkpoint['best_val_loss']


In [ ]:
import torch.optim as optim
# Initialize the model
model = FNPG(args.graph_channels,args.batch_size,args.fea_litho,args.head,args.drop).to(device)
#model = Unet_fused(args.graph_channels,args.batch_size,args.fea_litho,args.head,args.drop).to(device)
#model = UNet()

model.to(device)
count_params(model)

num_epochs = 135
train_size = args.sam_num
iterations = num_epochs * (train_size // args.batch_size) # 计算迭代次数

# Define loss function and optimizer
criterion = CustomLossWithAdaptiveDerivativePenaltyAndWavelet()
optimizer = optim.Adam(list(model.parameters())+list(criterion.parameters()), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=iterations)
first_train  = False
if first_train:
  # Lists to store metrics
  train_loss_list = []
  val_loss_list = []
  test_loss_list = []
else:
  train_loss_list = np.load('train_loss_epoch65_重建.npy').tolist()
  val_loss_list = np.load('val_loss_epoch65_重建.npy').tolist()
  test_loss_list = np.load('test_loss_epoch65_重建.npy').tolist()
#模型保存
savename = f'{STAGE}' + '50' + '重建_50epoch.pt' #读取的模型
filename_4read = args.save_model_path + savename
savename = f'{STAGE}' + '100' + '重建_100epoch.pt' #要保存的模型
filename = args.save_model_path + savename
# 读取最好的模型
model, optimizer, best_epoch, best_loss = load_best_model(model, optimizer, filename_4read)
#print(f"Loaded best model from epoch {best_epoch} with validation loss {best_loss}")

device used in conv_obj： cuda
device used in conv_obj： cuda
device used in conv_obj： cuda
9826212


<ipython-input-12-8e1390ddf749>:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path)


## Train loop of FNPG without data augmentation

In [ ]:
# Training loop
best_val_loss = float('inf')
out_list = []
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for a in tqdm(train_loader):
        a = a.to(device)
        a.x = a.x.view(args.batch_size,len(args.fea_litho),args.sam_len)
        optimizer.zero_grad()
        output = model(a)
        out_list.append(output)
        #plot_inputs(output.detach().cpu(),[''],range(0,512))
        loss = criterion(output.view(args.batch_size,-1,args.sam_len), a.y.view(args.batch_size,-1,args.sam_len))
        loss.backward()
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()
        # gradient clip
        #torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for a in tqdm(val_loader):
            a = a.to(device)
            a.x = a.x.view(args.batch_size,len(args.fea_litho),args.sam_len)
            output = model(a)
            val_loss += criterion(output.view(args.batch_size,-1,args.sam_len), a.y.view(args.batch_size,-1,args.sam_len)).item()

    # Evaluation
    model.eval()
    test_loss = 0
    with torch.no_grad():
      for a in tqdm(test_loader):
        a = a.to(device)
        a.x = a.x.view(args.batch_size,len(args.fea_litho),args.sam_len)
        output = model(a)
        test_loss += criterion(output.view(args.batch_size,-1,args.sam_len), a.y.view(args.batch_size,-1,args.sam_len)).item()

    # Calculate average losses
    train_loss /= len(train_loader)
    val_loss /= len(val_loader)
    test_loss /= len(test_loader)
    # Store metrics
    train_loss_list.append(train_loss)
    val_loss_list.append(val_loss)
    test_loss_list.append(test_loss)
    print(f'Epoch {epoch+1}/{num_epochs}|Train Loss={train_loss:.4e}|Val Loss={val_loss:.4e}|Test Loss: {test_loss:.4e}')

    if isinstance(criterion, CustomLossWithAdaptiveDerivativePenaltyAndWavelet):
      if epoch % 10 == 0:
          alpha, beta, gamma = criterion.get_alpha_beta_gamma()
          print(f"Epoch {epoch}, Alpha: {alpha:.4f}, Beta: {beta:.4f}, Gamma: {gamma:.4f}")

    # Check if this is the best model so far
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(model, optimizer, epoch, best_val_loss, filename)
        print(f"New best model saved at epoch {epoch}")

# Load the best model after training

model, optimizer, best_epoch, best_loss = load_best_model(model, optimizer, filename)
print(f"Loaded best model from epoch {best_epoch} with validation loss {best_loss}")

print("Training complete!")

100%|██████████| 1/1 [00:00<00:00,  5.98it/s]


Epoch 1/110|Train Loss=5.7540e-01|Val Loss=5.7140e-01|Test Loss: 8.7721e-01
Epoch 0, Alpha: 0.1000, Beta: 0.0100, Gamma: 0.1001
New best model saved at epoch 0


100%|██████████| 1/1 [00:00<00:00,  5.33it/s]


Epoch 2/110|Train Loss=5.7494e-01|Val Loss=5.7221e-01|Test Loss: 8.7626e-01


100%|██████████| 1/1 [00:00<00:00,  5.77it/s]


...

Epoch 108/110|Train Loss=5.7100e-01|Val Loss=6.3112e-01|Test Loss: 9.2510e-01


100%|██████████| 1/1 [00:00<00:00,  5.77it/s]


Epoch 109/110|Train Loss=5.6719e-01|Val Loss=5.7005e-01|Test Loss: 8.8468e-01


100%|██████████| 1/1 [00:00<00:00,  5.69it/s]
<ipython-input-14-8e1390ddf749>:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path)


Epoch 110/110|Train Loss=5.6920e-01|Val Loss=5.8135e-01|Test Loss: 8.9978e-01
Loaded best model from epoch 88 with validation loss 0.5651903413236141
Training complete!


## Train Loop


In [ ]:
# Training loop
best_test_loss = float('inf')
out_list = []
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for a in tqdm(train_loader):
        a = a.to(device)
        #print(a.x.shape)
        a.x = a.x.view(args.batch_size,len(args.fea_litho),args.sam_len)
        optimizer.zero_grad()
        output = model(a)
        out_list.append(output)
        #plot_inputs(output.detach().cpu(),[''],range(0,512))
        loss = criterion(output.view(args.batch_size, -1, args.sam_len),
                      a.y.view(args.batch_size, -1, args.sam_len))
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

        try:
            train_loss += loss.item()

            if np.isnan(train_loss):
                if train_loss_list:
                    train_loss = trian_loss_list[-1]
                else:
                    train_loss = 1
        except Exception as e:
            print(f"An error occurred: {e}")


    # Evaluation
    model.eval()
    test_loss = 0
    with torch.no_grad():
      for a in tqdm(test_loader):
        a = a.to(device)
        a.x = a.x.view(args.batch_size,len(args.fea_litho),args.sam_len)
        output = model(a)
        try:
            test_loss += criterion(output.view(args.batch_size, -1, args.sam_len),
                                  a.y.view(args.batch_size, -1, args.sam_len)).item()

            if np.isnan(test_loss):
                if test_loss_list:
                    test_loss = test_loss_list[-1]
                else:
                    test_loss = 1

        except Exception as e:
            print(f"An error occurred: {e}")
    # Calculate average losses
    train_loss /= len(train_loader)
    test_loss /= len(test_loader)
    # Store metrics
    train_loss_list.append(train_loss)
    test_loss_list.append(test_loss)
    print(f'Epoch {epoch+1}/{num_epochs}|Train Loss={train_loss:.4e}|Test Loss: {test_loss:.4e}')

    if isinstance(criterion, CustomLossWithAdaptiveDerivativePenaltyAndWavelet):
      if epoch % 10 == 0:
          alpha, beta, gamma = criterion.get_alpha_beta_gamma()
          print(f"Epoch {epoch}, Alpha: {alpha:.4f}, Beta: {beta:.4f}, Gamma: {gamma:.4f}")

    # Check if this is the best model so far
    if test_loss < best_test_loss:
        best_test_loss = test_loss
        save_checkpoint(model, optimizer, epoch, best_test_loss, filename)
        print(f"New best model saved at epoch {epoch}")

    scheduler.step()
# Load the best model after training

model, optimizer, best_epoch, best_loss = load_best_model(model, optimizer, filename)
print(f"Loaded best model from epoch {best_epoch} with validation loss {best_loss}")

print("Training complete!")


100%|██████████| 3/3 [00:12<00:00,  4.01s/it]


Epoch 1/135|Train Loss=1.7910e-01|Test Loss: 1.9956e-01
Epoch 0, Alpha: 0.1000, Beta: 0.0100, Gamma: 0.1003
New best model saved at epoch 0


100%|██████████| 3/3 [00:12<00:00,  4.31s/it]


Epoch 2/135|Train Loss=1.8184e-01|Test Loss: 2.0192e-01


100%|██████████| 3/3 [00:12<00:00,  4.06s/it]


...

Epoch 43/135|Train Loss=nan|Test Loss: 2.3285e-03
New best model saved at epoch 42


 33%|███▎      | 1/3 [00:23<00:47, 23.57s/it]

An error occurred: name 'trian_loss_list' is not defined


 33%|███▎      | 1/3 [00:28<00:56, 28.18s/it]


KeyboardInterrupt: 

In [ ]:
# 假设你的训练循环已经完成，现在有了这些列表

# 将列表转换为 NumPy 数组
train_loss_array = np.array(train_loss_list[:-4])
val_loss_array = np.array(val_loss_list[:-4])
test_loss_array = np.array(test_loss_list[:-4])


# 保存数组
np.save('train_loss_epoch104_重建.npy', train_loss_array)
np.save('val_loss_epoch104_重建.npy', val_loss_array)
np.save('test_loss_epoch104_重建.npy', test_loss_array)

# 之后，你可以这样加载数据：
# loaded_train_loss = np.load('train_loss_epoch20_vnila.npy')
# loaded_val_loss = np.load('val_loss_epoch20_vnila.npy')
# loaded_test_loss = np.load('test_loss_epoch20_vnila.npy')

##for drawing

In [ ]:
import torch
import glob
import os
file_pattern_x = '/content/drive/MyDrive/logcompletion/data/Teamdata/complete_data_for_PIModel/test/Test_x_all_complete_*.pt'
file_pattern_y = '/content/drive/MyDrive/logcompletion/data/Teamdata/complete_data_for_PIModel/test/Test_y_all_complete_*.pt'
# Get all files matching the pattern
x_files = glob.glob(file_pattern_x)
y_files = glob.glob(file_pattern_y)
# Sort the files to ensure they are in the correct order
x_files.sort()
y_files.sort()
# List to store individual tensors
tensor_list_x = []
tensor_list_y = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# file就是每口井
for file in x_files:
    # 查看tensor_list_x的测井顺序，用于提取井名
    print(f"Loading tensor from file: {file}")  # Print the file name
    tensor = torch.load(file,map_location=torch.device(device))
    well_len = tensor
    tensor_list_x.append(tensor)#20+22
print('\n')

for file in y_files:
    print(f"Loading tensor from file: {file}")  # Print the file name
    tensor = torch.load(file,map_location=torch.device(device))
    tensor_list_y.append(tensor)

print('\n')
# Concatenate all tensors along the first dimension
combined_tensor_x = torch.cat(tensor_list_x, dim=0)
combined_tensor_y = torch.cat(tensor_list_y, dim=0) # 这个y是GNN模型的结果

print(f"Combined tensor shape: {combined_tensor_x.shape}")
print(f"Combined tensor shape: {combined_tensor_y.shape}")

Loading tensor from file: /content/drive/MyDrive/logcompletion/data/Teamdata/complete_data_for_PIModel/test/Test_x_all_complete_31.3-2.pt
Loading tensor from file: /content/drive/MyDrive/logcompletion/data/Teamdata/complete_data_for_PIModel/test/Test_x_all_complete_31.3-3.pt


Loading tensor from file: /content/drive/MyDrive/logcompletion/data/Teamdata/complete_data_for_PIModel/test/Test_y_all_complete_31.3-2.pt
Loading tensor from file: /content/drive/MyDrive/logcompletion/data/Teamdata/complete_data_for_PIModel/test/Test_y_all_complete_31.3-3.pt


Combined tensor shape: torch.Size([42, 4, 512])
Combined tensor shape: torch.Size([42, 4, 512])


<ipython-input-34-f8f3359c901e>:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  tensor = torch.load(file,map_location=torch.device(device))
<ipython-input-34-f8f3359c901e>

In [ ]:
len(tensor_list_x[0])

20

In [ ]:
# 获得FNPG模型的combined_tensor_y
STAGE = 'Reconstruction'
from types import SimpleNamespace
args = SimpleNamespace(
    batch_size=1,
    sam_num=512,
    epochs=500,
    save_model_path=f'/content/drive/MyDrive/logcompletion/model/ctra_exp_{STAGE}/',
    lossfigpath=F'/content/drive/MyDrive/logcompletion/figure/ctra_exp_{STAGE}/',
    sam_len=512,
    graph_channels=[512,256,128,64,32],
    data_norm_type='',
    LR=0.001,
    fea_litho=['GR', 'RHOB', 'NPHI', 'DTC'],
    need_maskd=True,
    filePath='/content/drive/MyDrive/logcompletion/data/Teamdata/standard_wells/data2/',
    loss_name=f'./loss_{STAGE}/loss_init.npz',
    dfVal_path=f'/content/drive/MyDrive/logcompletion/data/Teamdata/standard_wells/minish_area/mini_val_standard_{STAGE}.csv',
    dfTrain_path=f'/content/drive/MyDrive/logcompletion/data/Teamdata/standard_wells/minish_area/mini_train_standard_{STAGE}.csv',
    dfTest_path=f'/content/drive/MyDrive/logcompletion/data/Teamdata/standard_wells/minish_area/mini_test_standard_{STAGE}.csv',
    inference_mode=False,
    filt_size=19,
    drop=0,
    head=1
)
model = FNPG(args.graph_channels,args.batch_size,args.fea_litho,args.head,args.drop).to(device)


model.to(device)

num_epochs = 135
train_size = args.sam_num
iterations = num_epochs * (train_size // args.batch_size) # 计算迭代次数
import torch.optim as optim

# Define loss function and optimizer
criterion = CustomLossWithAdaptiveDerivativePenaltyAndWavelet()
optimizer = optim.Adam(list(model.parameters())+list(criterion.parameters()), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=iterations)

# 读取最好的模型
savename = f'{STAGE}' + '50' + '重建_50epoch.pt'
filename = args.save_model_path + savename
model, optimizer, best_epoch, best_loss = load_best_model(model, optimizer, filename)
#print(f"Loaded best model from epoch {best_epoch} with validation loss {best_loss}")

device used in conv_obj： cuda
device used in conv_obj： cuda
device used in conv_obj： cuda


<ipython-input-32-8e1390ddf749>:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path)


In [ ]:
# 0 index 指代第一口井 31/3-2
well_len_start = len(tensor_list_x[0])
well_len = len(tensor_list_x[1])


In [ ]:
# 获得FNPG模型的combined_tensor_y
modelPredictionList = []
inputDataList = []
for i in range(well_len_start,well_len + well_len_start):
    data = Data(input_sam=combined_tensor_x[i:i+1],label=None)
    y = model(data)
    modelPredictionList.append(y)
    inputDataList.append(data.x)

modelPredictionList_pt = torch.cat(modelPredictionList)
inputDataList_pt = torch.cat(inputDataList)
# 得到完整井
modelPredictionList_pt = modelPredictionList_pt.transpose(0,1).reshape(4,-1)
inputDataList_pt = inputDataList_pt.transpose(0,1).reshape(4,-1)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime
def plot_results(data, predictions, target, log_names, depth, fig_size=(20, 24), dpi=300):
    # Reverse the depth array
    #depth = depth[::-1]

    # Set the style for the plots
    sns.set_style("whitegrid")
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']

    # Move tensors to CPU and convert to NumPy arrays
    data = data.cpu().numpy()
    n_input_logs = data.shape[1]

    if target is not None:
        predictions = predictions.cpu().numpy()
        target = target.cpu().numpy()
        n_predicted_logs = predictions.shape[1]

        # Create subplots
        fig, axs = plt.subplots(1, n_input_logs + n_predicted_logs + 1, figsize=fig_size, dpi=dpi, sharey=True)
        fig.suptitle("Well Log Prediction Results", fontsize=24, fontweight='bold')

        # Plot input logs
        for i in range(n_input_logs):
            axs[i].plot(data[0, i, :], depth, label='Actual', color='black', linewidth=0.5)
            axs[i].set_title(log_names[i], fontsize=16)
            axs[i].set_xlabel(log_names[i] + ' Value', fontsize=12)
            axs[i].grid(True, which='both', linestyle='--', linewidth=0.5)
            axs[i].tick_params(axis='both', which='major', labelsize=10)

        # Plot predicted vs actual logs, leaving gaps for target values equal to 5
        for i in range(n_predicted_logs):
            predictions_plot = np.where(target[0, i, :] == 5, np.nan, predictions[0, i, :])
            axs[n_input_logs + i].plot(target[0, i, :], depth, label='Actual', color='black', linewidth=0.5)
            axs[n_input_logs + i].plot(predictions[0, i, :], depth, label='Predicted', color='red', linewidth=0.5, linestyle='--')
            axs[n_input_logs + i].set_title(log_names[n_input_logs + i], fontsize=16)
            axs[n_input_logs + i].set_xlabel(log_names[n_input_logs + i] + ' Value', fontsize=12)
            axs[n_input_logs + i].legend(fontsize=10)
            axs[n_input_logs + i].grid(True, which='both', linestyle='--', linewidth=0.5)
            axs[n_input_logs + i].tick_params(axis='both', which='major', labelsize=10)

        # Set common y-label
        fig.text(0.04, 0.5, 'Depth (m)', va='center', rotation='vertical', fontsize=14)

        # Plot error distribution and cross-plot
        error = predictions[0, 0, :] - target[0, 0, :]
        ax_hist = axs[-1]
        sns.histplot(error, kde=True, ax=ax_hist)
        ax_hist.set_title('Error Distribution', fontsize=16)
        ax_hist.set_xlabel('Error', fontsize=12)
        ax_hist.set_ylabel('Frequency', fontsize=12)

        # Add statistical information to the error distribution plot
        mean_error = np.mean(error)
        std_error = np.std(error)
        rmse = np.sqrt(np.mean(error**2))
        r2 = stats.pearsonr(predictions[0, 0, :], target[0, 0, :])[0]**2

        stats_text = f'Mean: {mean_error:.2f}\nStd: {std_error:.2f}\nRMSE: {rmse:.2f}\nR²: {r2:.2f}'
        ax_hist.text(0.95, 0.95, stats_text, transform=ax_hist.transAxes, fontsize=10,
                    verticalalignment='top', horizontalalignment='right',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

        # Add a cross-plot as an inset axis in the error distribution subplot
        ax_cross = ax_hist.inset_axes([0.05, 0.6, 0.4, 0.35])
        ax_cross.scatter(target[0, 0, :], predictions[0, 0, :], alpha=0.5, s=1)
        ax_cross.set_xlabel('Actual', fontsize=8)
        ax_cross.set_ylabel('Predicted', fontsize=8)
        ax_cross.set_title('Cross-plot', fontsize=10)

        # Add a 45-degree reference line
        lims = [
            np.min([ax_cross.get_xlim(), ax_cross.get_ylim()]),
            np.max([ax_cross.get_xlim(), ax_cross.get_ylim()]),
        ]
        ax_cross.plot(lims, lims, 'r--', alpha=0.75, zorder=0)
        ax_cross.set_aspect('equal')
        ax_cross.tick_params(axis='both', which='major', labelsize=6)
    else:
        # Create subplots
        fig, axs = plt.subplots(1, n_input_logs, figsize=fig_size, dpi=dpi, sharey=True)
        fig.suptitle("Well Log Reconstruction Results", fontsize=24, fontweight='bold')
        #data = np.flip(data)
        #predictions = torch.flip(predictions, dims=[-1])
        # Plot input logs
        for i in range(n_input_logs):
            axs[i].plot(data[0, i, :], depth, label='Actual', color='black', linewidth=0.5)
            axs[i].plot(predictions[0, i, :], depth, label='Reconstruction', color='red', linewidth=0.5, linestyle='--')
            axs[i].set_title(log_names[i], fontsize=16)
            axs[i].set_xlabel(log_names[i] + ' Value', fontsize=12)
            axs[i].grid(True, which='both', linestyle='--', linewidth=0.5)
            axs[i].tick_params(axis='both', which='major', labelsize=10)

        # Set common y-label
        fig.text(0.04, 0.5, 'Depth (m)', va='center', rotation='vertical', fontsize=14)

    # Set y-axis limits for all subplots
    for ax in axs:
        ax.set_ylim(depth.max(), depth.min())

    plt.tight_layout(rect=[0.05, 0.03, 1, 0.95])
    return fig


log_names = ['GR', 'RHOB', 'NPHI', 'DTC']
depth = np.arange(0, modelPredictionList_pt.shape[-1])
depth_512 = np.arange(0,512)
# 512 切片
#i= 1
#fig = plot_results(combined_tensor_x[i:i+1].detach().cpu(), combined_tensor_y[i:i+1].detach().cpu(), None, log_names, depth_512)

# 完整井
fig = plot_results(inputDataList_pt.detach().cpu().unsqueeze(0), modelPredictionList_pt.detach().cpu().unsqueeze(0), None, log_names, depth)

# 获取当前时间并格式化为字符串
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"/content/drive/MyDrive/logcompletion/figure/31.3-2_完整重建图{timestamp}.pdf"
#plt.savefig(filename, format='pdf', dpi=600, bbox_inches='tight')

#plt.close(fig)